# 🧬 Panacée — Phase 2 : Fine-tuning toxicité (Kaggle)

**Phase 2** : Fine-tuning toxicité (12 endpoints Tox21, ROC-AUC supervisé cliniquement).  
Charge l'encodeur Phase 1 → produit `best_toxicity_model.pth`.

### Chaîne :
```
Phase 1 → sovereign_encoder_v1.pth  ← INPUT ici (optionnel mais recommandé)
         ↓
Phase 2 (ce notebook)  → best_toxicity_model.pth
         ↓
Phase 3 (kaggle_phase3.ipynb)
```

### Avant de lancer :
1. Dashboard local + ngrok actifs
2. (Optionnel) Ajouter `sovereign_encoder_v1.pth` en **Input** Kaggle → changer `PHASE1_CKPT`
3. Remplir `NGROK_URL` et `TOKEN`
4. ▶▶ Run All

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  CONFIGURATION — modifier ces lignes ONLY    ║
# ╚══════════════════════════════════════════════╝

NGROK_URL  = "https://unintimidated-feelingly-reva.ngrok-free.dev"          # URL publique de votre dashboard
TOKEN         = "panacee_kaggle_2026"            # == PANACEE_INGEST_TOKEN dans .env
RUN_NAME      = "kaggle_phase2_run01"            # Nom affiché dans le dashboard
N_EPOCHS      = 300  # GPU: 50-100, CPU: 10-20
BATCH_SIZE = 512   # taille de batch GPU — plus grand = GPU mieux saturé (réduire si OOM)
MAX_MOLECULES = None                             # None = tout Tox21, ou ex: 2000 pour test

# Checkpoint Phase 1 (optionnel mais améliore beaucoup les résultats).
# Si vous avez uploadé sovereign_encoder_v1.pth en Input Kaggle :
#   PHASE1_CKPT = "/kaggle/input/panacee-phase1/sovereign_encoder_v1.pth"
# Sinon laisser None → encodeur initialisé aléatoirement (Phase 2 autonome).
PHASE1_CKPT = None

print(f"Run         : {RUN_NAME}")
print(f"Epochs      : {N_EPOCHS}")
print(f"Phase 1     : {PHASE1_CKPT or 'non fourni (encodeur aléatoire)'}")

In [ ]:
# ── Vérification connexion dashboard ────────────────────────────────────────
import json
import urllib.request

try:
    r = urllib.request.urlopen(NGROK_URL.rstrip('/') + '/api/health', timeout=8)
    print(f"✅ Dashboard joignable : {json.loads(r.read())}")
except Exception as e:
    print(f"❌ Dashboard NON joignable : {e}")
    raise SystemExit("Corrigez NGROK_URL avant de continuer.")

In [ ]:
# ── Vérification checkpoint Phase 1 (si fourni) ─────────────────────────────
from pathlib import Path

if PHASE1_CKPT:
    if Path(PHASE1_CKPT).exists():
        size_mb = Path(PHASE1_CKPT).stat().st_size / 1e6
        print(f"✅ Encodeur Phase 1 : {PHASE1_CKPT} ({size_mb:.1f} MB)")
    else:
        print(f"❌ Checkpoint Phase 1 introuvable : {PHASE1_CKPT}")
        print("   → Vérifiez le chemin dans /kaggle/input/...")
        raise SystemExit("Checkpoint Phase 1 manquant.")
else:
    print("⚠️  Pas de Phase 1 → encodeur initialisé aléatoirement.")
    print("   Résultats corrects mais moins bons qu'avec Phase 1 pré-entraînée.")

In [ ]:
# ── Variables d'environnement ────────────────────────────────────────────────
import os

os.environ["PANACEE_PUSH_URL"]   = NGROK_URL
os.environ["PANACEE_PUSH_TOKEN"] = TOKEN
os.environ["PANACEE_PUSH_RUN"]   = RUN_NAME
os.environ["PYTHONIOENCODING"]   = "utf-8"
# Performance : 1 worker DataLoader par cœur CPU + préchargement
# → le CPU prépare les batchs pendant que le GPU calcule (anti-famine).
os.environ["PANACEE_NUM_WORKERS"] = str(os.cpu_count() or 4)

print("Variables ✓")

In [ ]:
# ── Dépendances ──────────────────────────────────────────────────────────────
import subprocess
import sys


def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=True)

pip("torch-geometric", "rdkit", "deepchem")
print("Dépendances ✓")

In [ ]:
# ── Cloner le repo ──────────────────────────────────────────────────────────
CLONE_DIR = Path("/kaggle/working/panacee")
REPO_URL  = "https://github.com/jumoras0000/savh_gnn.git"

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--ff-only"], check=False)

PROJET = CLONE_DIR / "Projet_Panacee"
if str(PROJET) not in sys.path:
    sys.path.insert(0, str(PROJET))
print(f"Projet : {PROJET}")

In [ ]:
# ── Répertoire de sauvegarde ────────────────────────────────────────────────
# Toujours 'phase2' → Phase 3 trouvera best_toxicity_model.pth automatiquement
SAVE_DIR = "/kaggle/working/checkpoints/phase2"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
print(f"Checkpoints → {SAVE_DIR}")
print(f"  Modèle final : {SAVE_DIR}/best_toxicity_model.pth")

In [ ]:
# ── Lancer Phase 2 ──────────────────────────────────────────────────────────
import os as _os

_os.chdir(str(PROJET))

sys.argv = [
    "run_phase2.py",
    "--download",
    "--save_dir",  SAVE_DIR,
    "--epochs",    str(N_EPOCHS),
    "--batch_size", str(BATCH_SIZE),
]
if PHASE1_CKPT:
    sys.argv += ["--pretrained_model", PHASE1_CKPT]
if MAX_MOLECULES:
    sys.argv += ["--max_molecules", str(MAX_MOLECULES)]

print(f"Phase 2 → run={RUN_NAME}, save={SAVE_DIR}")
print(f"Push  → {os.environ['PANACEE_PUSH_URL']}")
print()

from run_phase2 import main

main()

In [ ]:
# ── Export du checkpoint → Output Kaggle ────────────────────────────────────
import shutil

OUTPUT_DIR = "/kaggle/output/panacee-phase2"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

ckpt_src = f"{SAVE_DIR}/best_toxicity_model.pth"
if Path(ckpt_src).exists():
    shutil.copy(ckpt_src, f"{OUTPUT_DIR}/best_toxicity_model.pth")
    size_mb = Path(ckpt_src).stat().st_size / 1e6
    print(f"✅ Checkpoint exporté : {OUTPUT_DIR}/best_toxicity_model.pth ({size_mb:.1f} MB)")
else:
    print(f"❌ Checkpoint introuvable : {ckpt_src}")

print()
print("=" * 60)
print("ÉTAPE SUIVANTE : Phase 3")
print("  1. Onglet Output → télécharger best_toxicity_model.pth")
print("  2. OU : Save Version → Output → 'Create dataset from output'")
print("  3. Ajouter comme Input dans le notebook kaggle_phase3.ipynb")
print("="*60)